In [39]:
import pandas as pd
import os
from glob import glob

In [40]:
paper_nodes = pd.read_csv("../../outputs/final/paper_nodes.csv")
entity_nodes = pd.read_csv("../../outputs/final/entity_nodes.csv")

print("Paper nodes:", len(paper_nodes))
print("Entity nodes:", len(entity_nodes))

Paper nodes: 2529
Entity nodes: 108816


In [41]:
## entity name → entity_id map
entity_map = dict(
    zip(entity_nodes["name"], entity_nodes["node_id"])
)

print("Entity map size:", len(entity_map))

Entity map size: 108816


In [42]:
## Triplet files

TRIPLETS_FOLDER = "../../../Scientific_Novelty_Detection/Triplets/"
triplet_files = glob(os.path.join(TRIPLETS_FOLDER, "**", "*_triplets.csv"), recursive=True)

print("Triplet files found:", len(triplet_files))

edges = []

Triplet files found: 17


In [43]:
## Normalize entity same way as entity_nodes
def normalize_entity(text):
    if pd.isna(text):
        return None
    text = str(text).strip().lower()
    text = text.replace('"""', '"')
    text = text.replace("''", "'")
    text = " ".join(text.split())
    return text

In [44]:
# Process all triplet files
paper_id_set = set(paper_nodes["node_id"])

# Process all triplet files
for file in triplet_files:

    # Extract split from parent folder
    folder_name = os.path.basename(os.path.dirname(file)).upper()

    if folder_name == "NOVEL_PAPERS":
        split = "NOVEL"
    elif folder_name == "BLOGS":
        split = "BLOG"
    else:
        split = "SKG"

    # Extract domain from filename
    filename = os.path.basename(file)
    domain = ''.join([c for c in filename.split("_")[0].upper() if not c.isdigit()])

    df = pd.read_csv(file)

    required_cols = ["sub", "obj", "pred", "paper_ID"]
    if not all(col in df.columns for col in required_cols):
        continue

    for _, row in df.iterrows():

        local_id = row["paper_ID"]
        global_id = f"{split}_{domain}_{local_id}"

        if global_id not in paper_id_set:
            continue

        source_year = paper_nodes.loc[
            paper_nodes["node_id"] == global_id, "year"
        ].values[0]

        sub = normalize_entity(row["sub"])
        obj = normalize_entity(row["obj"])
        predicate = row["pred"]

        for entity in [sub, obj]:

            if entity in entity_map:

                edges.append({
                    "source": global_id,
                    "target": entity_map[entity],
                    "predicate": predicate,
                    "year": source_year
                })

In [45]:
print(len(edges))

335921


In [46]:
# Create DataFrame
knowledge_edges = pd.DataFrame(edges)

print("Raw knowledge edges:", len(knowledge_edges))

# Remove duplicates
knowledge_edges = knowledge_edges.drop_duplicates()

print("After deduplication:", len(knowledge_edges))

Raw knowledge edges: 335921
After deduplication: 286708


In [47]:
# Sanity Checks
print("\nSANITY CHECKS")

print("Duplicate edges:",
      knowledge_edges.duplicated().sum())

print("Missing year values:",
      knowledge_edges["year"].isnull().sum())

assert knowledge_edges["source"].isin(
    paper_nodes["node_id"]
).all()

assert knowledge_edges["target"].isin(
    entity_nodes["node_id"]
).all()


SANITY CHECKS
Duplicate edges: 0
Missing year values: 0


In [48]:
## Save
knowledge_edges.to_csv("../../outputs/final/knowledge_edges.csv", index=False)

print("\nSaved: knowledge_edges.csv")


Saved: knowledge_edges.csv


In [49]:
print("Papers with no edges:",
      len(set(paper_nodes["node_id"]) - set(knowledge_edges["source"])))

Papers with no edges: 0


In [50]:
paper_node_empty_year = paper_nodes[paper_nodes["year"].isna()]
paper_node_empty_year

,node_id,node_type,title,domain,split,year,openalex_id,cited_by_count,score,method


In [51]:
knowledge_edges

,source,target,predicate,year
0,NOVEL_DIA_0,E_a83c35a686,pre-trained on,2021.0
1,NOVEL_DIA_0,E_a56f5c8422,pre-trained on,2021.0
2,NOVEL_DIA_0,E_a83c35a686,evaluated on,2021.0
3,NOVEL_DIA_0,E_84e314f125,evaluated on,2021.0
4,NOVEL_DIA_0,E_b575844c21,has,2021.0
...,...,...,...,...
335913,SKG_SUM_26,E_144cd83446,produced,2020.0
335915,SKG_SUM_26,E_90b7c64f69,produced,2020.0
335916,SKG_SUM_26,E_24642ab5c3,with,2020.0
335917,SKG_SUM_26,E_90b7c64f69,without,2020.0


In [52]:
knowledge_edges.merge(
    paper_nodes[["node_id", "domain"]],
    left_on="source",
    right_on="node_id"
)["domain"].value_counts()

domain
MT     124114
SA      47384
QA      43585
DIA     31556
SUM     25155
PAR     13237
NLI      1677
Name: count, dtype: int64